In [1]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("../data/raw/ratings.csv")

ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26024289 entries, 0 to 26024288
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 794.2 MB


In [3]:
print(ratings["movieId"].sample(10, random_state=42).tolist())

[163, 55118, 3499, 2571, 4973, 56286, 3360, 3755, 5577, 4343]


In [ ]:
# movies_meta = pd.read_csv(
#     "../data/raw/movies_metadata.csv",
#     low_memory=False
# )

# # Remove corrupted rows
# movies_meta = movies_meta[movies_meta['id'].str.isdigit()]
# movies_meta['id'] = movies_meta['id'].astype(int)

In [ ]:
# movies_meta.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45463 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45463 non-null  object 
 1   belongs_to_collection  4491 non-null   object 
 2   budget                 45463 non-null  object 
 3   genres                 45463 non-null  object 
 4   homepage               7779 non-null   object 
 5   id                     45463 non-null  int32  
 6   imdb_id                45446 non-null  object 
 7   original_language      45452 non-null  object 
 8   original_title         45463 non-null  object 
 9   overview               44509 non-null  object 
 10  popularity             45460 non-null  object 
 11  poster_path            45077 non-null  object 
 12  production_companies   45460 non-null  object 
 13  production_countries   45460 non-null  object 
 14  release_date           45376 non-null  object 
 15  revenue

In [8]:
from pandas.core.common import random_state
print(movies["id"].sample(10, random_state=42).tolist())

['411405', '42492', '12143', '9976', '46761', '268725', '62297', '184885', '156145', '306745']


In [14]:
links = pd.read_csv(
    '../data/raw/links.csv',
    dtype={'movieId': 'int64', 'tmdbId': 'float64'}
)

links.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45843 entries, 0 to 45842
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  45843 non-null  int64  
 1   imdbId   45843 non-null  int64  
 2   tmdbId   45624 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 1.0 MB


In [17]:
# Validate Overlap before merge
ratings_unique = ratings["movieId"].nunique()
links_unique = ratings["movieId"].nunique()

common_id = set(ratings["movieId"]) & set(links["movieId"])

print(f"\nUnique ratings movieIds: {ratings_unique}")
print(f"\nUnique links movieIds: {links_unique}")
print(f"\nOverlap %: {100* len(common_id) / ratings_unique:.4f}%")
print(f"\nOverlap %: {100* len(common_id) / links_unique:.4f}%")


Unique ratings movieIds: 45115

Unique links movieIds: 45115

Overlap %: 100.0000%

Overlap %: 100.0000%


In [19]:
# Merge ratings to links
ratings_with_tmdb = ratings.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="left")

missing_ratio = ratings_with_tmdb["tmdbId"].isna().mean()
print(f"\nMissing tmdbId ratio after merge: {missing_ratio}")

# Decision Gate
if missing_ratio > 0.01:
    print("Warning: More than 1% ratings missing tmdbId. Investigate.")
else:
    print("Mapping integrity accepted")

# Drop unmatch
ratings_clean = ratings_with_tmdb.dropna(subset=["tmdbId"]).copy()
ratings_clean["tmdbId"] = ratings_clean["tmdbId"].astype(int)

print(f"\nOriginal rows: {len(ratings):,}")
print(f"Retained rows: {len(ratings_clean):,}")
print(f"Retention: {100* len(ratings_clean) / len(ratings):.2f}%")


#  Saved processed file
ratings_clean.to_csv("../data/processed/ratings_clean.csv", index=False)
print("Saved: data/processed/ratings_clean.csv")


Missing tmdbId ratio after merge: 0.0005188614374824996
Mapping integrity accepted

Original rows: 26,024,289
Retained rows: 26,010,786
Retention: 99.95%
Saved: data/processed/ratings_clean.csv


In [33]:
movies_meta = pd.read_csv("../data/raw/movies_metadata.csv", low_memory=False)
print("Initial shape:", movies_meta.shape)
print("Initial dtypes:\n", movies_meta.dtypes)

Initial shape: (45466, 24)
Initial dtypes:
 adult                     object
belongs_to_collection     object
budget                    object
genres                    object
homepage                  object
id                        object
imdb_id                   object
original_language         object
original_title            object
overview                  object
popularity                object
poster_path               object
production_companies      object
production_countries      object
release_date              object
revenue                  float64
runtime                  float64
spoken_languages          object
status                    object
tagline                   object
title                     object
video                     object
vote_average             float64
vote_count               float64
dtype: object


In [38]:
credits = pd.read_csv("../data/raw/credits.csv")

print("Credits Shape:", credits.shape)
print(credits.head())

Credits Shape: (45476, 3)
                                                cast  \
0  [{'cast_id': 14, 'character': 'Woody (voice)',...   
1  [{'cast_id': 1, 'character': 'Alan Parrish', '...   
2  [{'cast_id': 2, 'character': 'Max Goldman', 'c...   
3  [{'cast_id': 1, 'character': "Savannah 'Vannah...   
4  [{'cast_id': 1, 'character': 'George Banks', '...   

                                                crew     id  
0  [{'credit_id': '52fe4284c3a36847f8024f49', 'de...    862  
1  [{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...   8844  
2  [{'credit_id': '52fe466a9251416c75077a89', 'de...  15602  
3  [{'credit_id': '52fe44779251416c91011acb', 'de...  31357  
4  [{'credit_id': '52fe44959251416c75039ed7', 'de...  11862  


In [ ]:
# Check duplicates in credits
duplicate_credits = credits['id'].value_counts()
print("Duplicate IDs in credits:", (duplicate_credits > 1).sum())

# If > 0:
credits = credits.drop_duplicates(subset=['id'], keep='first')

In [39]:
import ast

def save_parse(x):
    try:
        return ast.literal_eval(x)
    except :
        return []

credits["cast"] = credits["cast"].apply(save_parse)
credits["crew"] = credits["crew"].apply(save_parse)

# Validation
print("\nCast example (first row):", credits["cast"].iloc[0][:3])
print("Crew example (first row directors):", [c["name"] for c in credits["crew"].iloc[0] if c.get("job") == "Director"])


Cast example (first row): [{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'cast_id': 15, 'character': 'Buzz Lightyear (voice)', 'credit_id': '52fe4284c3a36847f8024f99', 'gender': 2, 'id': 12898, 'name': 'Tim Allen', 'order': 1, 'profile_path': '/uX2xVf6pMmPepxnvFWyBtjexzgY.jpg'}, {'cast_id': 16, 'character': 'Mr. Potato Head (voice)', 'credit_id': '52fe4284c3a36847f8024f9d', 'gender': 2, 'id': 7167, 'name': 'Don Rickles', 'order': 2, 'profile_path': '/h5BcaDMPRVLHLDzbQavec4xfSdt.jpg'}]
Crew example (first row directors): ['John Lasseter']


In [40]:
movies_full = movies_meta.merge(
    credits[["id", "cast", "crew"]],
    on="id",
    how="left"
)

print("After credit merge:", movies_full.shape)
print("Missing cast rows:", movies_full["cast"].isna().sum())
print("Missing crew rows:", movies_full["crew"].isna().sum())

After credit merge: (45468, 26)
Missing cast rows: 1
Missing crew rows: 1
